In [10]:
from pathlib import Path
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [11]:
# ============================================================
# DATASET PATH
# ============================================================

data_set_path = str(
    Path.home()
    / "Desktop"
    / "fighting-churn-learning"
    / "output"
    / "churn_dataset.csv"
)

# Data Prepration and listing

In [9]:
# ============================================================
# LISTING 5 — DATASET SUMMARY STATISTICS
# ============================================================

def dataset_stats(data_set_path):

    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    numeric_data = churn_data.apply(
        pd.to_numeric,
        errors="coerce"
    )

    stats = numeric_data.describe().T

    stats["skew"] = numeric_data.skew()

    stats["1%"] = numeric_data.quantile(0.01)
    stats["99%"] = numeric_data.quantile(0.99)

    stats["nonzero"] = (
        numeric_data.ne(0).sum()
        / len(numeric_data)
    )

    save_path = data_set_path.replace(
        ".csv",
        "_summarystats.csv"
    )

    stats.to_csv(save_path)

    print(f"Saved: {save_path}")

    return stats


# ============================================================
# LISTING 7.5 — FAT-TAIL SCORING
# ============================================================

def transform_skew_columns(data, columns):

    for col in columns:
        data[col] = np.log1p(data[col])


def transform_fattail_columns(data, columns):

    for col in columns:
        data[col] = np.arcsinh(data[col])


def fat_tail_scores(
    data_set_path,
    skew_thresh=4.0
):

    churn_data = pd.read_csv(
        data_set_path,
        index_col=[0, 1]
    )

    data_scores = churn_data.copy()

    if "is_churn" not in data_scores.columns:
        raise ValueError(
            "is_churn column is missing."
        )

    target = data_scores["is_churn"].copy()

    data_scores = data_scores.drop(
        columns=["is_churn"]
    )

    data_scores = data_scores.apply(
        pd.to_numeric,
        errors="coerce"
    )

    stat_path = data_set_path.replace(
        ".csv",
        "_summarystats.csv"
    )

    if not os.path.isfile(stat_path):
        raise FileNotFoundError(
            "Run dataset_stats first."
        )

    stats = pd.read_csv(
        stat_path,
        index_col=0
    )

    stats = stats.drop(
        index="is_churn",
        errors="ignore"
    )

    common_columns = [
        col
        for col in data_scores.columns
        if col in stats.index
    ]

    data_scores = data_scores[
        common_columns
    ]

    stats = stats.loc[
        common_columns
    ]

    # --------------------------------------------------------
    # Positive skew → log1p
    # --------------------------------------------------------

    skewed_columns = (
        (stats["skew"] > skew_thresh)
        &
        (stats["min"] >= 0)
    )

    transform_skew_columns(
        data_scores,
        skewed_columns[
            skewed_columns
        ].index
    )

    # --------------------------------------------------------
    # Fat tails with negative values → asinh
    # --------------------------------------------------------

    fattail_columns = (
        (stats["skew"].abs() > skew_thresh)
        &
        (stats["min"] < 0)
    )

    transform_fattail_columns(
        data_scores,
        fattail_columns[
            fattail_columns
        ].index
    )

    # --------------------------------------------------------
    # Standardize
    # --------------------------------------------------------

    mean_vals = data_scores.mean()
    std_vals = data_scores.std()

    # Prevent division by zero
    valid_columns = std_vals[
        std_vals > 0
    ].index

    data_scores = data_scores[
        valid_columns
    ]

    mean_vals = mean_vals[
        valid_columns
    ]

    std_vals = std_vals[
        valid_columns
    ]

    data_scores = (
        data_scores - mean_vals
    ) / std_vals

    data_scores["is_churn"] = target

    # --------------------------------------------------------
    # Save scores
    # --------------------------------------------------------

    score_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    data_scores.to_csv(
        score_path
    )

    print(f"Saved: {score_path}")

    # --------------------------------------------------------
    # Save transformation parameters
    # --------------------------------------------------------

    param_df = pd.DataFrame(
        index=valid_columns
    )

    param_df["skew_score"] = (
        skewed_columns.reindex(
            valid_columns
        ).fillna(False)
    )

    param_df["fattail_score"] = (
        fattail_columns.reindex(
            valid_columns
        ).fillna(False)
    )

    param_df["mean"] = mean_vals
    param_df["std"] = std_vals

    param_path = data_set_path.replace(
        ".csv",
        "_score_params.csv"
    )

    param_df.to_csv(
        param_path
    )

    print(f"Saved: {param_path}")

    return data_scores


# ============================================================
# LISTING 6.4 — FIND CORRELATED METRIC GROUPS
# ============================================================

def find_metric_groups(
    data_set_path,
    group_corr_thresh=0.55
):

    score_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    data_scores = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    metrics = data_scores.drop(
        columns=["is_churn"],
        errors="ignore"
    )

    corr = metrics.corr()

    # Numerical cleanup
    corr = corr.fillna(0)

    np.fill_diagonal(
        corr.values,
        1.0
    )

    # --------------------------------------------------------
    # Book method:
    #
    # dissimilarity = 1 - correlation
    #
    # NOT 1 - abs(correlation)
    # --------------------------------------------------------

    dissimilarity = 1.0 - corr

    dissimilarity = (
        dissimilarity
        + dissimilarity.T
    ) / 2.0

    np.fill_diagonal(
        dissimilarity.values,
        0.0
    )

    condensed = squareform(
        dissimilarity.values,
        checks=False
    )

    linkage_matrix = linkage(
        condensed,
        method="single"
    )

    cluster_ids = fcluster(
        linkage_matrix,
        t=1.0 - group_corr_thresh,
        criterion="distance"
    )

    cluster_series = pd.Series(
        cluster_ids,
        index=metrics.columns,
        name="group"
    )

    # --------------------------------------------------------
    # Relabel largest groups first
    # --------------------------------------------------------

    group_sizes = (
        cluster_series
        .value_counts()
        .sort_values(
            ascending=False
        )
    )

    relabel = {
        old_group: new_group
        for new_group, old_group
        in enumerate(
            group_sizes.index,
            start=1
        )
    }

    cluster_series = (
        cluster_series
        .map(relabel)
    )

    # --------------------------------------------------------
    # Loading matrix
    # --------------------------------------------------------

    groups = sorted(
        cluster_series.unique()
    )

    load_mat = pd.DataFrame(
        0.0,
        index=metrics.columns,
        columns=[
            f"group_{group}"
            for group in groups
        ]
    )

    for group in groups:

        members = cluster_series[
            cluster_series == group
        ].index

        group_size = len(members)

        if group_size == 1:

            weight = 1.0

        else:

            weight = (
                1.0
                /
                (
                    np.sqrt(
                        group_corr_thresh
                    )
                    * group_size
                )
            )

        load_mat.loc[
            members,
            f"group_{group}"
        ] = weight

    # --------------------------------------------------------
    # Save loading matrix
    # --------------------------------------------------------

    load_path = data_set_path.replace(
        ".csv",
        "_load_mat.csv"
    )

    load_mat.to_csv(
        load_path
    )

    print(f"Saved: {load_path}")

    # --------------------------------------------------------
    # Save group membership
    # --------------------------------------------------------

    group_metric_df = (
        cluster_series
        .rename("group")
        .to_frame()
    )

    group_metric_path = (
        data_set_path.replace(
            ".csv",
            "_groupmets.csv"
        )
    )

    group_metric_df.to_csv(
        group_metric_path
    )

    print(
        f"Saved: {group_metric_path}"
    )

    return load_mat, group_metric_df


# ============================================================
# LISTING 6.3 — APPLY METRIC GROUPS
# ============================================================

def apply_metric_groups(data_set_path):

    score_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    load_path = data_set_path.replace(
        ".csv",
        "_load_mat.csv"
    )

    data_scores = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    load_mat = pd.read_csv(
        load_path,
        index_col=0
    )

    target = data_scores[
        "is_churn"
    ].copy()

    metric_scores = data_scores.drop(
        columns=["is_churn"]
    )

    # Exact column order required for matrix multiplication
    metric_scores = metric_scores[
        load_mat.index
    ]

    grouped_values = (
        metric_scores.values
        @ load_mat.values
    )

    grouped_scores = pd.DataFrame(
        grouped_values,
        index=metric_scores.index,
        columns=load_mat.columns
    )

    grouped_scores[
        "is_churn"
    ] = target

    save_path = data_set_path.replace(
        ".csv",
        "_groupscore.csv"
    )

    grouped_scores.to_csv(
        save_path
    )

    print(f"Saved: {save_path}")

    return grouped_scores


# ============================================================
# LISTING 6.5 — ORDERED CORRELATION MATRIX
# ============================================================

def ordered_correlation_matrix(
    data_set_path
):

    score_path = data_set_path.replace(
        ".csv",
        "_scores.csv"
    )

    load_path = data_set_path.replace(
        ".csv",
        "_load_mat.csv"
    )

    scores = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    load_mat = pd.read_csv(
        load_path,
        index_col=0
    )

    metric_scores = scores.drop(
        columns=["is_churn"],
        errors="ignore"
    )

    metric_scores = metric_scores[
        load_mat.index
    ]

    corr = metric_scores.corr()

    save_path = data_set_path.replace(
        ".csv",
        "_ordered_correlation_matrix.csv"
    )

    corr.to_csv(
        save_path
    )

    print(f"Saved: {save_path}")

    return corr


# ============================================================
# LISTING 8.1 — COMPLETE DATA PREPARATION PIPELINE
# ============================================================

def prepare_data(
    data_set_path,
    group_corr_thresh=0.55,
    skew_thresh=4.0
):

    print("\n1. Calculating dataset statistics...")
    dataset_stats(
        data_set_path
    )

    print("\n2. Calculating fat-tail metric scores...")
    fat_tail_scores(
        data_set_path,
        skew_thresh=skew_thresh
    )

    print("\n3. Finding correlated metric groups...")
    find_metric_groups(
        data_set_path,
        group_corr_thresh=group_corr_thresh
    )

    print("\n4. Applying metric groups...")
    apply_metric_groups(
        data_set_path
    )

    print("\n5. Creating ordered correlation matrix...")
    ordered_correlation_matrix(
        data_set_path
    )

    print("\nChapter 8 data preparation complete.")


# ============================================================
# RUN LISTING 8.1
# ============================================================

prepare_data(
    data_set_path=data_set_path,
    group_corr_thresh=0.55,
    skew_thresh=4.0
)


1. Calculating dataset statistics...
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_summarystats.csv

2. Calculating fat-tail metric scores...
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_scores.csv
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_score_params.csv

3. Finding correlated metric groups...
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_load_mat.csv
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_groupmets.csv

4. Applying metric groups...
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_groupscore.csv

5. Creating ordered correlation matrix...
Saved: /Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_ordered_correlation_matrix.csv

Chapter 8 data preparation complete.


# Logistic regression analysis

In [12]:
# ============================================================
# ROBUST CHURN TARGET CONVERSION
# ============================================================

def convert_churn_target(series):

    print("\nRaw is_churn values:")
    print(series.value_counts(dropna=False))

    # --------------------------------------------------------
    # Already boolean
    # --------------------------------------------------------

    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    # --------------------------------------------------------
    # Numeric 0 / 1
    # --------------------------------------------------------

    if pd.api.types.is_numeric_dtype(series):

        values = set(
            series.dropna().unique()
        )

        if not values.issubset({0, 1, 0.0, 1.0}):
            raise ValueError(
                "Numeric is_churn contains values other than 0/1:\n"
                f"{sorted(values)}"
            )

        return series.astype(int).astype(bool)

    # --------------------------------------------------------
    # Strings such as:
    #
    # True / False
    # true / false
    # t / f
    # 1 / 0
    # --------------------------------------------------------

    cleaned = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapping = {
        "true": True,
        "false": False,
        "t": True,
        "f": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False
    }

    unknown = set(
        cleaned.unique()
    ) - set(mapping.keys())

    if unknown:
        raise ValueError(
            "Unknown values found in is_churn:\n"
            f"{unknown}"
        )

    return cleaned.map(mapping).astype(bool)


# ============================================================
# PREPARE DATA
# ============================================================

def prepare_data(
    data_set_path,
    ext="_groupscore",
    as_retention=True
):

    score_path = data_set_path.replace(
        ".csv",
        f"{ext}.csv"
    )

    if not os.path.isfile(score_path):
        raise FileNotFoundError(
            "Grouped score dataset not found.\n"
            "Run Listing 8.1 / Listing 6.3 first.\n"
            f"Expected:\n{score_path}"
        )

    grouped_data = pd.read_csv(
        score_path,
        index_col=[0, 1]
    )

    if "is_churn" not in grouped_data.columns:
        raise ValueError(
            '"is_churn" column is missing.'
        )

    # --------------------------------------------------------
    # CORRECT target conversion
    # --------------------------------------------------------

    is_churn = convert_churn_target(
        grouped_data["is_churn"]
    )

    print("\nConverted churn target:")
    print(
        is_churn.value_counts()
    )

    # --------------------------------------------------------
    # Book models retention
    # --------------------------------------------------------

    if as_retention:

        # True = retention
        # False = churn
        y = ~is_churn

    else:

        # True = churn
        # False = retention
        y = is_churn

    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    X = grouped_data.drop(
        columns=["is_churn"]
    )

    X = X.apply(
        pd.to_numeric,
        errors="coerce"
    )

    if X.isna().any().any():

        bad_columns = X.columns[
            X.isna().any()
        ].tolist()

        raise ValueError(
            "NaN/non-numeric feature values found in:\n"
            f"{bad_columns}"
        )

    # --------------------------------------------------------
    # Verify target has exactly TWO classes
    # --------------------------------------------------------

    unique_classes = np.unique(y)

    print(
        "\nTarget classes:",
        unique_classes
    )

    if len(unique_classes) != 2:
        raise ValueError(
            "Target must contain exactly two classes.\n"
            f"Found: {unique_classes}"
        )

    return X, y


# ============================================================
# SIGMOID
# ============================================================

def sigmoid(x):

    x = np.asarray(
        x,
        dtype=float
    )

    x = np.clip(
        x,
        -709,
        709
    )

    return 1.0 / (
        1.0 + np.exp(-x)
    )


# ============================================================
# CALCULATE IMPACTS
# ============================================================

def calculate_impacts(retain_reg):

    intercept = float(
        retain_reg.intercept_[0]
    )

    coefficients = (
        retain_reg.coef_[0]
    )

    average_retain = float(
        sigmoid(intercept)
    )

    one_stdev_retains = sigmoid(
        intercept + coefficients
    )

    one_stdev_impact = (
        one_stdev_retains
        - average_retain
    )

    return (
        one_stdev_impact,
        average_retain
    )


# ============================================================
# LOAD GROUP DESCRIPTIONS
# ============================================================

def load_group_descriptions(
    data_set_path,
    model_feature_names
):

    group_path = data_set_path.replace(
        ".csv",
        "_groupmets.csv"
    )

    if not os.path.isfile(group_path):

        return {
            feature: "(group membership unavailable)"
            for feature in model_feature_names
        }

    group_df = pd.read_csv(
        group_path,
        index_col=0
    )

    if "group" not in group_df.columns:

        return {
            feature: "(group membership unavailable)"
            for feature in model_feature_names
        }

    descriptions = {}

    for feature in model_feature_names:

        if not feature.startswith("group_"):

            descriptions[feature] = feature
            continue

        try:

            group_number = int(
                feature.replace(
                    "group_",
                    ""
                )
            )

        except ValueError:

            descriptions[feature] = feature
            continue

        members = group_df.index[
            group_df["group"] == group_number
        ].tolist()

        descriptions[feature] = (
            ", ".join(
                map(str, members)
            )
            if members
            else "(no metrics found)"
        )

    return descriptions


# ============================================================
# SAVE REGRESSION SUMMARY
# ============================================================

def save_regression_summary(
    data_set_path,
    retain_reg,
    feature_names,
    ext=""
):

    (
        one_stdev_impact,
        average_retain
    ) = calculate_impacts(
        retain_reg
    )

    descriptions = (
        load_group_descriptions(
            data_set_path,
            feature_names
        )
    )

    rows = []

    for (
        feature,
        coefficient,
        impact
    ) in zip(
        feature_names,
        retain_reg.coef_[0],
        one_stdev_impact
    ):

        rows.append(
            {
                "group_metric_offset": feature,
                "weight": coefficient,
                "retain_impact": impact,
                "group_metrics":
                    descriptions.get(
                        feature,
                        feature
                    )
            }
        )

    rows.append(
        {
            "group_metric_offset":
                "offset",

            "weight":
                float(
                    retain_reg.intercept_[0]
                ),

            "retain_impact":
                average_retain,

            "group_metrics":
                "(baseline)"
        }
    )

    coef_df = pd.DataFrame(
        rows
    )

    save_path = (
        data_set_path.replace(
            ".csv",
            f"_logreg_summary{ext}.csv"
        )
    )

    coef_df.to_csv(
        save_path,
        index=False
    )

    print(
        f"\nSaved coefficients to:\n"
        f"{save_path}"
    )

    return coef_df


# ============================================================
# SAVE MODEL
# ============================================================

def save_regression_model(
    data_set_path,
    retain_reg,
    feature_names,
    ext=""
):

    pickle_path = (
        data_set_path.replace(
            ".csv",
            f"_logreg_model{ext}.pkl"
        )
    )

    model_package = {

        "model":
            retain_reg,

        "feature_names":
            list(feature_names),

        "target":
            "retention"
    }

    with open(
        pickle_path,
        "wb"
    ) as fid:

        pickle.dump(
            model_package,
            fid
        )

    print(
        f"\nSaved model pickle to:\n"
        f"{pickle_path}"
    )

    return pickle_path


# ============================================================
# SAVE PREDICTIONS
# ============================================================

def save_dataset_predictions(
    data_set_path,
    retain_reg,
    X,
    ext=""
):

    probabilities = (
        retain_reg.predict_proba(
            X
        )
    )

    print(
        "\nModel classes:",
        retain_reg.classes_
    )

    class_to_index = {
        class_value: i
        for i, class_value
        in enumerate(
            retain_reg.classes_
        )
    }

    churn_index = (
        class_to_index[False]
    )

    retain_index = (
        class_to_index[True]
    )

    predict_df = pd.DataFrame(
        {
            "churn_prob":
                probabilities[
                    :,
                    churn_index
                ],

            "retain_prob":
                probabilities[
                    :,
                    retain_index
                ]
        },
        index=X.index
    )

    predict_path = (
        data_set_path.replace(
            ".csv",
            f"_predictions{ext}.csv"
        )
    )

    predict_df.to_csv(
        predict_path
    )

    print(
        f"\nSaved predictions to:\n"
        f"{predict_path}"
    )

    return predict_df


# ============================================================
# LISTING 8.2
# ============================================================

def logistic_regression_analysis(
    data_set_path
):

    print(
        "\nLoading grouped training dataset..."
    )

    X, y = prepare_data(
        data_set_path,
        ext="_groupscore",
        as_retention=True
    )

    print(
        f"\nRows: {len(X):,}"
    )

    print(
        f"Model features: {X.shape[1]:,}"
    )

    print(
        f"Churn observations: {(~y).sum():,}"
    )

    print(
        f"Retention observations: {y.sum():,}"
    )

    print(
        "\nRetention rate:"
        f" {y.mean():.4%}"
    )

    print(
        "Churn rate:"
        f" {(~y).mean():.4%}"
    )

    # --------------------------------------------------------
    # Book-style L1 logistic regression
    # --------------------------------------------------------

    retain_reg = LogisticRegression(
        fit_intercept=True,
        solver="liblinear",
        penalty="l1",
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    print(
        "\nFitting logistic regression..."
    )

    retain_reg.fit(
        X,
        y
    )

    regression_summary = (
        save_regression_summary(
            data_set_path,
            retain_reg,
            feature_names=X.columns
        )
    )

    save_regression_model(
        data_set_path,
        retain_reg,
        feature_names=X.columns
    )

    predictions = (
        save_dataset_predictions(
            data_set_path,
            retain_reg,
            X
        )
    )

    print(
        "\nListing 8.2 complete."
    )

    return (
        retain_reg,
        X,
        y,
        regression_summary,
        predictions
    )


# ============================================================
# RUN
# ============================================================

(
    retain_reg,
    X,
    y,
    regression_summary,
    predictions
) = logistic_regression_analysis(
    data_set_path
)


# ============================================================
# DISPLAY RESULTS
# ============================================================

print(
    "\nRegression summary:"
)

display(
    regression_summary
)


print(
    "\nFirst predictions:"
)

display(
    predictions.head()
)


print(
    "\nProbability sum check:"
)

display(
    predictions[
        ["churn_prob", "retain_prob"]
    ]
    .sum(axis=1)
    .describe()
)


Loading grouped training dataset...

Raw is_churn values:
is_churn
f    32252
t      645
Name: count, dtype: int64

Converted churn target:
is_churn
False    32252
True       645
Name: count, dtype: int64

Target classes: [False  True]

Rows: 32,897
Model features: 2
Churn observations: 645
Retention observations: 32,252

Retention rate: 98.0393%
Churn rate: 1.9607%

Fitting logistic regression...

Saved coefficients to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_logreg_summary.csv

Saved model pickle to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_logreg_model.pkl

Model classes: [False  True]

Saved predictions to:
/Users/toor/Desktop/fighting-churn-learning/output/churn_dataset_predictions.csv

Listing 8.2 complete.

Regression summary:


/opt/anaconda3/envs/fightchurn/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/opt/anaconda3/envs/fightchurn/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


,group_metric_offset,weight,retain_impact,group_metrics
0,group_1,0.515015,0.006734,like_per_month
1,group_2,0.209844,0.003156,account_tenure
2,offset,4.063226,0.983097,(baseline)



First predictions:


,,churn_prob,retain_prob
account_id,observation_date,,
82,2020-02-09,0.016792,0.983208
122,2020-02-09,0.044465,0.955535
184,2020-02-09,0.038442,0.961558
262,2020-02-09,0.030621,0.969379
291,2020-02-09,0.033051,0.966949



Probability sum check:


count    32897.0
mean         1.0
std          0.0
min          1.0
25%          1.0
50%          1.0
75%          1.0
max          1.0
dtype: float64